In [58]:
import re
import pandas as pd

def find_sheet(sheet_names):
    for name in sheet_names:
        if 'primary' in name.lower() and '3' in name and 'char' in name.lower():
            return name
    return None

In [59]:
def find_column(columns, keywords):
    for col in columns:
        col_str = str(col).lower()
        if all(k.lower() in col_str for k in keywords):
            return col
    return None


In [60]:
def find_start_row(file_path, sheet_name):
    preview = pd.read_excel(file_path, sheet_name=sheet_name, nrows=30, header=None)

    for i, row in preview.iterrows():
        row_str = ' '.join([str(x).lower() for x in row])

        if (
                'primary' in row_str and
                'diagnosis' in row_str and
                'emergency' in row_str
        ):
            return i

    return None


In [61]:
def load_year(file_path):
    xls = pd.ExcelFile(file_path)
    sheet_name = find_sheet(xls.sheet_names)
    print("find excel name:", sheet_name)

    if sheet_name is None:
        raise ValueError(f"Sheet not found in {file_path}")

    start_row = find_start_row(file_path, sheet_name)

    if start_row is None:
        raise ValueError(f"Start row not found in {file_path}")

    df = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=start_row)
    print("Start row:", start_row)

    # 更稳的 Code 列
    df = df.rename(columns={df.columns[0]: "Code"})

    # 防 NaN 列名
    df.columns = df.columns.fillna('').astype(str)
    df.columns = df.columns.str.strip()

    # 调试：打印所有列名
    print(f"All columns in {file_path}:")
    print(df.columns.tolist())

    # 自动找列 - 优先用列名，失败则用位置（第2列通常是诊断）
    diag_col = find_column(df.columns, ['primary', 'diagnosis'])
    if diag_col is None:
        # 如果没找到诊断列，用第二列（索引1）
        diag_col = df.columns[1]
        print(f"  → diag_col not found by name, using column at index 1: {diag_col}")

    emergency_col = find_column(df.columns, ['emergency'])

    print(f"  → diag_col: {diag_col}")
    print(f"  → emergency_col: {emergency_col}")

    if emergency_col is None:
        print("ERROR: Emergency column not found. Columns:", df.columns.tolist())
        raise ValueError(f"Emergency column not found in {file_path}")

    df = df[['Code', diag_col, emergency_col]].dropna()
    print(f"  → After dropna: {len(df)} rows")

    df = df.rename(columns={
        diag_col: 'Diagnosis',
        emergency_col: 'Emergency'
    })

    # 年份提取（更稳）
    match = re.search(r'(\d{4})', file_path)
    if match:
        df['Year'] = int(match.group(1))
    else:
        raise ValueError(f"Year not found in {file_path}")

    return df


In [62]:

import os

folder_path = './data'

all_files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx') and not f.startswith('~$')]
df_list = []
for file in all_files:
    file_path = os.path.join(folder_path, file)
    df_year = load_year(file_path)
    df_year = df_year.dropna(how='all').dropna(subset=['Code', 'Diagnosis', 'Emergency', 'Year'], how='any')
    df_list.append(df_year)

# 合并所有年份前，先去掉空表
clean_df_list = [df for df in df_list if not df.empty]

# 合并所有年份
df_all = pd.concat(clean_df_list, ignore_index=True)
df_all = df_all.dropna(subset=['Code', 'Diagnosis', 'Emergency', 'Year']).reset_index(drop=True)
df_all.head()

# 导出新的 Excel 文件
output_file = './hospital_diagnosis_combined.xlsx'

df_all['Emergency'] = pd.to_numeric(df_all['Emergency'], errors='coerce')

year_summary = (
    df_all.groupby('Year', as_index=False)['Emergency']
    .sum()
    .sort_values('Year')
)

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_all.to_excel(writer, sheet_name='CombinedData', index=False)
    year_summary.to_excel(writer, sheet_name='YearSummary', index=False)

output_file

df = pd.read_excel('./hospital_diagnosis_combined.xlsx', sheet_name='CombinedData')
df.head()


find excel name: Primary diagnosis - 3 char
Start row: 18
All columns in ./data\hosp-epis-stat-admi-diag-2012-13-tab.xlsx:
['Code', 'Primary diagnosis: 3 character code and description', 'Finished consultant episodes', 'Admissions', 'Male', 'Female', 'Gender unknown', 'Emergency', 'Waiting list', 'Planned', 'Other admission method', 'Mean time waited', 'Median time waited', 'Mean length of stay', 'Median length of stay', 'Mean age', 'Age 0', 'Age 1-4', 'Age 5-9', 'Age 10-14', 'Age 15', 'Age 16', 'Age 17', 'Age 18', 'Age 19', 'Age 20-24', 'Age 25-29', 'Age 30-34', 'Age 35-39', 'Age 40-44', 'Age 45-49', 'Age 50-54', 'Age 55-59', 'Age 60-64', 'Age 65-69', 'Age 70-74', 'Age 75-79', 'Age 80-84', 'Age 85-89', 'Age 90+', 'Day case', 'FCE Bed Days']
  → diag_col: Primary diagnosis: 3 character code and description
  → emergency_col: Emergency
  → After dropna: 1624 rows
find excel name: Primary Diagnosis 3 Character
Start row: 15
All columns in ./data\hosp-epis-stat-admi-diag-2013-14-tab.xlsx:

,Code,Diagnosis,Emergency,Year
0,A00,Cholera ...,7.0,2012
1,A01,Typhoid and paratyphoid fevers ...,240.0,2012
2,A02,Other salmonella infections ...,537.0,2012
3,A03,Shigellosis ...,95.0,2012
4,A04,Other bacterial intestinal infections ...,6963.0,2012
